# Intuition test: linear algebra and calculus

Questions come first. For each one, write your prediction in the cell marked `Prediction`, then run the code cell below it. Answers are at the end, after every question. Do not scroll down early.

Seeded and self-contained.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(42)
np.set_printoptions(precision=3, suppress=True)

## Q1. A matrix and the unit circle

$A = \begin{bmatrix} 3 & 0 \\ 0 & 0.5 \end{bmatrix}$. What shape does the unit circle become? Where does the point $(0, 1)$ land? What is the area of the result, as a multiple of the circle's area?

Prediction: *(write here)*

In [ ]:
A = np.array([[3.0, 0.0], [0.0, 0.5]])
theta = np.linspace(0, 2 * np.pi, 200)
circle = np.vstack([np.cos(theta), np.sin(theta)])
out = A @ circle
print("(0, 1) lands at:", A @ np.array([0.0, 1.0]))
print("det:", np.linalg.det(A))
plt.figure(figsize=(4, 3))
plt.plot(*circle, alpha=0.4)
plt.plot(*out)
plt.gca().set_aspect("equal")
plt.show()

## Q2. A rotation

$R$ rotates by 90 degrees anticlockwise. Predict $R^2$, $R^4$, and $\det R$.

Prediction: *(write here)*

In [ ]:
R = np.array([[0.0, -1.0], [1.0, 0.0]])
print("R^2:\n", R @ R)
print("R^4:\n", np.linalg.matrix_power(R, 4))
print("det R:", np.linalg.det(R))

## Q3. Rank

$B = \begin{bmatrix} 1 & 2 & 3 \\ 2 & 4 & 6 \\ 1 & 0 & 1 \end{bmatrix}$. What is its rank? Is it invertible?

Prediction: *(write here)*

In [ ]:
B = np.array([[1.0, 2, 3], [2, 4, 6], [1, 0, 1]])
print("rank:", np.linalg.matrix_rank(B), " det:", np.linalg.det(B))

## Q4. Eigenvalues

For $S = \begin{bmatrix} 2 & 0 \\ 0 & 5 \end{bmatrix}$, write down the eigenvalues without computing. Then for $T = \begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix}$, guess one eigenvector and its eigenvalue.

Prediction: *(write here)*

In [ ]:
print("S:", np.linalg.eigvalsh(np.array([[2.0, 0], [0, 5]])))
vals, vecs = np.linalg.eigh(np.array([[2.0, 1], [1, 2]]))
print("T eigenvalues:", vals, "\nvectors:\n", vecs)

## Q5. A gradient by hand, then check it

Let $f(x, y) = x^2 y + \sin(x y)$ at $(x, y) = (1, 2)$. Compute $\partial f/\partial x$ and $\partial f/\partial y$ on paper. Then predict whether the gradient points towards larger or smaller $f$.

Hint: $\partial f/\partial x = 2xy + y\cos(xy)$ and $\partial f/\partial y = x^2 + x\cos(xy)$. Evaluate them yourself first.

Prediction: *(write the two numbers here)*

In [ ]:
# autodiff engine from notebook 04, repeated so this notebook stands alone
import math


class Value:
    def __init__(self, data, parents=()):
        self.data, self.grad, self._parents = float(data), 0.0, parents
        self._backward = lambda: None

    def __add__(self, o):
        o = o if isinstance(o, Value) else Value(o)
        out = Value(self.data + o.data, (self, o))

        def _b():
            self.grad += out.grad
            o.grad += out.grad

        out._backward = _b
        return out

    def __mul__(self, o):
        o = o if isinstance(o, Value) else Value(o)
        out = Value(self.data * o.data, (self, o))

        def _b():
            self.grad += o.data * out.grad
            o.grad += self.data * out.grad

        out._backward = _b
        return out

    def __pow__(self, k):
        out = Value(self.data**k, (self,))

        def _b():
            self.grad += k * self.data ** (k - 1) * out.grad

        out._backward = _b
        return out

    def sin(self):
        out = Value(math.sin(self.data), (self,))

        def _b():
            self.grad += math.cos(self.data) * out.grad

        out._backward = _b
        return out

    def backward(self):
        order, seen = [], set()

        def visit(v):
            if v not in seen:
                seen.add(v)
                for p in v._parents:
                    visit(p)
                order.append(v)

        visit(self)
        self.grad = 1.0
        for v in reversed(order):
            v._backward()


x, y = Value(1.0), Value(2.0)
f = x**2 * y + (x * y).sin()
f.backward()
print("autodiff:", x.grad, y.grad)

fn = lambda a, b: a**2 * b + np.sin(a * b)
h = 1e-6
print("numeric: ", (fn(1 + h, 2) - fn(1 - h, 2)) / (2 * h), (fn(1, 2 + h) - fn(1, 2 - h)) / (2 * h))
print("by hand: ", 2 * 1 * 2 + 2 * np.cos(2), 1 + np.cos(2))

## Q6. Learning rate

Minimise $f(x) = x^2$ from $x_0 = 1$ with gradient descent. After one step with learning rate 0.1, where are you? With learning rate 1.0, after 3 steps? With 1.1, after 3 steps?

Prediction: *(write here)*

In [ ]:
for lr in (0.1, 1.0, 1.1):
    xx, trail = 1.0, [1.0]
    for _ in range(3):
        xx -= lr * 2 * xx
        trail.append(xx)
    print(f"lr {lr}:", np.round(trail, 3))

## Q7. Collinear features

You fit least squares with two features where the second is exactly twice the first. How many solutions fit the data equally well? What does `lstsq` return?

Prediction: *(write here)*

In [ ]:
x1 = rng.uniform(-1, 1, size=50)
X = np.column_stack([x1, 2 * x1])
y = 3 * x1
w, _, rank, _ = np.linalg.lstsq(X, y, rcond=None)
print("rank:", rank, " weights:", w, " check w0 + 2*w1:", w[0] + 2 * w[1])

## Answers

Read these only after you have written your predictions.

**Q1.** An ellipse, stretched to 3 along x and squashed to 0.5 along y. $(0, 1)$ lands at $(0, 0.5)$. The determinant is 1.5, so the area is 1.5 times larger. A matrix can squash one direction and stretch another and still grow area overall.

**Q2.** $R^2 = -I$, $R^4 = I$, $\det R = 1$. Rotations preserve area and four quarter-turns get you home.

**Q3.** Rank 2. Row 2 is twice row 1, so the determinant is 0 and there is no inverse. If you said 3, you read the shape and not the content.

**Q4.** $S$ has eigenvalues 2 and 5, read straight off the diagonal. $T$ has eigenvector $[1, 1]/\sqrt{2}$ with eigenvalue 3, and $[1, -1]/\sqrt{2}$ with eigenvalue 1. Each row sums to 3, which is the giveaway for $[1, 1]$.

**Q5.** $\partial f/\partial x = 4 + 2\cos 2 \approx 3.168$ and $\partial f/\partial y = 1 + \cos 2 \approx 0.584$. Both are positive, so the gradient points towards larger $f$, and descent would go the other way. The autodiff and numerical values should match your hand result to about six digits.

**Q6.** Learning rate 0.1 gives $x = 0.8$. Rate 1.0 flips between $-1$ and $1$ forever and never converges. Rate 1.1 gives $-1.2, 1.44, -1.728$, growing 20% per step. The update multiplies $x$ by $(1 - 2\eta)$, so anything with $|1 - 2\eta| \ge 1$ fails. That is $\eta \ge 1$ here.

**Q7.** Infinitely many: any $w_0 + 2 w_1 = 3$ works. `lstsq` returns the one with the smallest norm, and reports rank 1. This is why regularisation is useful. What would change if the second feature were twice the first plus a tiny amount of noise?